In [25]:
import os 
from pathlib import Path 
import pandas as pd 
import numpy as np 

from lightgbm import LGBMClassifier

import warnings

In [26]:
warnings.filterwarnings("ignore")

In [2]:
os.chdir("..")

In [3]:
print(os.getcwd())

e:\project_archive\new project


In [4]:
# Load data

data_path = Path("data/processed")
if not data_path.exists():
    raise FileNotFoundError

X_train = pd.read_csv(data_path / "x_train.csv")
y_train = np.load(data_path / "y_train.npy")
print("X_train and y_train loaded successfully")

X_test = pd.read_csv(data_path / "x_test.csv")
y_test = np.load(data_path / "y_test.npy")
print("X_test and y_test loaded successfully")

X_train and y_train loaded successfully
X_test and y_test loaded successfully


In [27]:
import yaml 

config_path = Path("config")
with open(config_path / "model_params.yaml", "r") as f:
    config = yaml.safe_load(f)

print(f"config file loaded")
print(config["LightGBM"])

config file loaded
{'n_estimators': {'min': 100, 'max': 500}, 'learning_rate': {'min': 0.01, 'max': 0.3, 'log': True}, 'max_depth': {'min': 1, 'max': 15}, 'num_leaves': {'min': 15, 'max': 255}, 'min_child_samples': {'min': 5, 'max': 100}, 'min_child_weight': {'min': 0.001, 'max': 10, 'log': True}, 'subsample': {'min': 0.6, 'max': 1.0}, 'subsample_freq': {'min': 0, 'max': 10}, 'colsample_bytree': {'min': 0.6, 'max': 1.0}, 'reg_alpha': {'min': 0.0, 'max': 10.0}, 'reg_lambda': {'min': 0.1, 'max': 20.0, 'log': True}, 'min_split_gain': {'min': 0.0, 'max': 5.0}, 'objective': ['multiclass'], 'boosting_type': ['gbdt'], 'class_weight': ['balanced', None], 'verbosity': [-1]}


In [18]:
params = config["LightGBM"]
print(params)

{'n_estimators': {'min': 100, 'max': 500}, 'learning_rate': {'min': 0.01, 'max': 0.3, 'log': True}, 'max_depth': {'min': 1, 'max': 15}, 'num_leaves': {'min': 15, 'max': 255}, 'min_child_samples': {'min': 5, 'max': 100}, 'min_child_weight': {'min': '1e-3', 'max': 10, 'log': True}, 'subsample': {'min': 0.6, 'max': 1.0}, 'subsample_freq': {'min': 0, 'max': 10}, 'colsample_bytree': {'min': 0.6, 'max': 1.0}, 'reg_alpha': {'min': 0.0, 'max': 10.0}, 'reg_lambda': {'min': 0.1, 'max': 20.0, 'log': True}, 'min_split_gain': {'min': 0.0, 'max': 5.0}, 'objective': ['multiclass'], 'boosting_type': ['gbdt'], 'class_weight': ['balanced', None], 'verbosity': [-1]}


In [21]:
model = LGBMClassifier(**params)
print(model)

LGBMClassifier(boosting_type=['gbdt'], class_weight=['balanced', None],
               colsample_bytree={'max': 1.0, 'min': 0.6},
               learning_rate={'log': True, 'max': 0.3, 'min': 0.01},
               max_depth={'max': 15, 'min': 1},
               min_child_samples={'max': 100, 'min': 5},
               min_child_weight={'log': True, 'max': 10, 'min': '1e-3'},
               min_split_gain={'max': 5.0, 'min': 0.0},
               n_estimators={'max': 500, 'min': 100},
               num_leaves={'max': 255, 'min': 15}, objective=['multiclass'],
               reg_alpha={'max': 10.0, 'min': 0.0},
               reg_lambda={'log': True, 'max': 20.0, 'min': 0.1},
               subsample={'max': 1.0, 'min': 0.6},
               subsample_freq={'max': 10, 'min': 0}, verbosity=[-1])


In [28]:
def suggest_params(trial, config):
    cfg = config["LightGBM"]

    return {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            cfg["n_estimators"]["min"],
            cfg["n_estimators"]["max"],
            step=50,
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            cfg["learning_rate"]["min"],
            cfg["learning_rate"]["max"],
            log=cfg["learning_rate"]["log"],
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            cfg["max_depth"]["min"],
            cfg["max_depth"]["max"],
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            cfg["num_leaves"]["min"],
            cfg["num_leaves"]["max"],
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            cfg["min_child_samples"]["min"],
            cfg["min_child_samples"]["max"],
        ),

        "min_child_weight": trial.suggest_float(
            "min_child_weight",
            cfg["min_child_weight"]["min"],
            cfg["min_child_weight"]["max"],
            log=cfg["min_child_weight"]["log"],
        ),

        "subsample": trial.suggest_float(
            "subsample",
            cfg["subsample"]["min"],
            cfg["subsample"]["max"],
        ),

        "subsample_freq": trial.suggest_int(
            "subsample_freq",
            cfg["subsample_freq"]["min"],
            cfg["subsample_freq"]["max"],
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            cfg["colsample_bytree"]["min"],
            cfg["colsample_bytree"]["max"],
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            cfg["reg_alpha"]["min"],
            cfg["reg_alpha"]["max"],
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            cfg["reg_lambda"]["min"],
            cfg["reg_lambda"]["max"],
            log=cfg["reg_lambda"]["log"],
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            cfg["min_split_gain"]["min"],
            cfg["min_split_gain"]["max"],
        ),

        "objective": cfg["objective"][0],
        "boosting_type": cfg["boosting_type"][0],
        "class_weight": trial.suggest_categorical(
            "class_weight",
            cfg["class_weight"],
        ),
        "verbosity": cfg["verbosity"][0],
        "random_state": 42,
        "n_jobs": -1,
    }

In [32]:
import optuna
import wandb
from pprint import pprint

from sklearn.model_selection import StratifiedKFold, cross_val_score

def objective(trial, X, y, config, run):

    params = suggest_params(
        trial=trial,
        config=config
    )

    model = LGBMClassifier(
        **params
        )

    cv_strategy = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    scores = cross_val_score(
        estimator=model,
        X=X,
        y=y,
        cv=cv_strategy,
        scoring="f1_macro",
        n_jobs=-1,
    )

    score = scores.mean()

    run.log(
        {
            "trial": trial.number,
            "f1_macro_avg": score,
        }
    )

    return score

In [33]:
# train.py
from sklearn.metrics import f1_score

with wandb.init(
        project="student-drop-enroll-grad-preds",
        name="LightGBM-Optuna-v1",
        group="LightGBM",
        tags=["LightGBM", "Optuna"]
    ) as run:
     
      study = optuna.create_study(
            direction='maximize'
      )

      study.optimize(
            lambda trial: objective(
                  trial,
                  X_train,
                  y_train,
                  config=config,
                  run=run
            ),
            n_trials=80
      )

      best_params = study.best_params

      model = LGBMClassifier(
            **best_params
      )

      model.fit(X_train, y_train)

      pred = model.predict(X_test)

      f1 = f1_score(
            y_test,
            pred,
            average='macro'
      )

      run.summary["best_cv_score"] = study.best_value
      run.summary["test_f1_macro"] = f1
      run.summary["best_params"] = best_params



[I 2026-06-26 14:28:43,682] A new study created in memory with name: no-name-1e1b3790-4186-4032-bce3-f0ca5678743c
[I 2026-06-26 14:28:51,907] Trial 0 finished with value: 0.6884012402930986 and parameters: {'n_estimators': 450, 'learning_rate': 0.0490988065832325, 'max_depth': 14, 'num_leaves': 49, 'min_child_samples': 47, 'min_child_weight': 0.11873810270305714, 'subsample': 0.7697950598906176, 'subsample_freq': 9, 'colsample_bytree': 0.9245589386311492, 'reg_alpha': 8.129499536212473, 'reg_lambda': 4.700035809437311, 'min_split_gain': 4.596058782453739, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.6884012402930986.
[I 2026-06-26 14:28:57,774] Trial 1 finished with value: 0.6996097745200595 and parameters: {'n_estimators': 100, 'learning_rate': 0.02911438027465665, 'max_depth': 5, 'num_leaves': 104, 'min_child_samples': 71, 'min_child_weight': 0.32288522659376906, 'subsample': 0.8403543674735792, 'subsample_freq': 5, 'colsample_bytree': 0.7721395578693059, 'reg_alpha': 3

f1_macro_avg,▅▆▁▆▆▇▇▇▇▇▇▇▇▅▇▇█▆▇▅▇█▆▇▇▇▆█▇▇▇▆▅▇▇████▇
trial,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
best_cv_score,0.72327
f1_macro_avg,0.71451
test_f1_macro,0.714
trial,79


In [24]:
print(type(config["LightGBM"]["min_child_weight"]["min"]))
print(config["LightGBM"]["min_child_weight"]["min"])

<class 'str'>
1e-3


In [34]:
study.best_params

{'n_estimators': 350,
 'learning_rate': 0.06086040424372452,
 'max_depth': 13,
 'num_leaves': 71,
 'min_child_samples': 20,
 'min_child_weight': 1.3909909425890912,
 'subsample': 0.9178950463687575,
 'subsample_freq': 5,
 'colsample_bytree': 0.932415659641114,
 'reg_alpha': 5.646192552076192,
 'reg_lambda': 5.306450868169528,
 'min_split_gain': 0.22045490805655238,
 'class_weight': 'balanced'}

In [35]:
study.best_value

0.7232714277357142

In [36]:
save_result_path = Path("artifacts/best_params")

save_result_path.mkdir(parents=True, exist_ok=True)

result = {
    "model_name": "LightGBM",
    "best_trial": study.best_trial.number,
    "best_params": study.best_params,
    "best_value": float(study.best_value)
}

with open(save_result_path / "lightgbm.yaml", "w") as f:
    yaml.safe_dump(result, f, sort_keys=False)
